<a href="https://colab.research.google.com/github/Karthikreddy1010/GenAI_Models/blob/main/Autoencoders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Auto Encoders**

In [2]:
"""
Standard Convolutional Autoencoder (AE) with Optimized Latent Space Visualization
Fast PCA and t-SNE computation using sampling and multiprocessing
FIXED: Automatic directory creation
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings
import os
warnings.filterwarnings('ignore')


# CREATE OUTPUT DIRECTORIES

output_dir = './outputs'
data_dir = './data'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

print(f"✓ Output directory: {os.path.abspath(output_dir)}")
print(f"✓ Data directory: {os.path.abspath(data_dir)}")

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("=" * 80)
print("STANDARD CONVOLUTIONAL AUTOENCODER WITH OPTIMIZED LATENT SPACE VISUALIZATION")
print("=" * 80)


# PART 1: STANDARD CONVOLUTIONAL AUTOENCODER


class ConvolutionalAutoencoder(nn.Module):
    """
    Standard Convolutional Autoencoder with encoder and decoder

    Architecture:
    - Encoder: 3 convolutional layers with stride=2 (downsampling)
    - Latent: 16×7×7 = 784 features (deterministic)
    - Decoder: 3 transpose convolutional layers (upsampling)
    """

    def __init__(self, latent_channels=16):
        super(ConvolutionalAutoencoder, self).__init__()
        self.latent_channels = latent_channels

        # ===== ENCODER =====
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),

            nn.Conv2d(in_channels=64, out_channels=latent_channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )

        # ===== DECODER =====
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(in_channels=latent_channels, out_channels=64,
                             kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),

            nn.ConvTranspose2d(in_channels=64, out_channels=32,
                             kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),

            nn.ConvTranspose2d(in_channels=32, out_channels=1,
                             kernel_size=3, stride=1, padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        """Encode input image to latent representation"""
        return self.encoder(x)

    def decode(self, z):
        """Decode latent representation to reconstructed image"""
        return self.decoder(z)

    def forward(self, x):
        """Forward pass: encode -> decode"""
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon, z


=
# PART 2: LOSS FUNCTION


def ae_loss(x_recon, x):
    """Reconstruction loss for autoencoder (MSE)"""
    return nn.MSELoss()(x_recon, x)


# PART 3: DATA LOADING


print("\n" + "=" * 80)
print("LOADING MNIST DATASET")
print("=" * 80)

transform = transforms.Compose([transforms.ToTensor()])

train_dataset = MNIST(root=data_dir, train=True, download=True, transform=transform)
test_dataset = MNIST(root=data_dir, train=False, download=True, transform=transform)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"✓ Training samples: {len(train_dataset)}")
print(f"✓ Test samples: {len(test_dataset)}")
print(f"✓ Batch size: {batch_size}")


# PART 4: TRAINING FUNCTION


def train_ae(model, train_loader, test_loader, epochs=15, lr=1e-3, device='cpu'):
    """Train the autoencoder"""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    train_losses = []
    test_losses = []

    model.to(device)

    for epoch in range(epochs):
        # Training Phase
        model.train()
        train_loss = 0
        for batch_idx, (data, _) in enumerate(train_loader):
            data = data.to(device)
            x_recon, _ = model(data)
            loss = ae_loss(x_recon, data)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # Testing Phase
        model.eval()
        test_loss = 0
        with torch.no_grad():
            for data, _ in test_loader:
                data = data.to(device)
                x_recon, _ = model(data)
                loss = ae_loss(x_recon, data)
                test_loss += loss.item()

        test_loss /= len(test_loader)
        test_losses.append(test_loss)

        if (epoch + 1) % 3 == 0:
            print(f"Epoch [{epoch+1:2d}/{epochs}] | Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f}")

    return train_losses, test_losses


# PART 5: TRAIN MODEL


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\n✓ Using device: {device}")

print("\n" + "=" * 80)
print("TRAINING STANDARD CONVOLUTIONAL AUTOENCODER")
print("=" * 80)

ae_model = ConvolutionalAutoencoder(latent_channels=16)
train_losses, test_losses = train_ae(ae_model, train_loader, test_loader,
                                      epochs=15, lr=1e-3, device=device)

print("\n✓ Training completed!")


# PART 6: EXTRACT LATENT REPRESENTATIONS


print("\n" + "=" * 80)
print("EXTRACTING LATENT REPRESENTATIONS")
print("=" * 80)

ae_model.eval()

# For training set - extract ALL for PCA/t-SNE
all_latents_train = []
all_labels_train = []

with torch.no_grad():
    for data, labels in train_loader:
        data = data.to(device)
        latent = ae_model.encode(data)
        all_latents_train.append(latent.cpu().numpy())
        all_labels_train.append(labels.numpy())

latents_train = np.concatenate(all_latents_train, axis=0)
labels_train = np.concatenate(all_labels_train, axis=0)

# Flatten latents
latents_train_flat = latents_train.reshape(latents_train.shape[0], -1)

# For test set
all_latents_test = []
all_labels_test = []

with torch.no_grad():
    for data, labels in test_loader:
        data = data.to(device)
        latent = ae_model.encode(data)
        all_latents_test.append(latent.cpu().numpy())
        all_labels_test.append(labels.numpy())

latents_test = np.concatenate(all_latents_test, axis=0)
labels_test = np.concatenate(all_labels_test, axis=0)
latents_test_flat = latents_test.reshape(latents_test.shape[0], -1)

print(f"✓ Latents shape: {latents_train.shape}")
print(f"✓ Latents flattened: {latents_train_flat.shape}")



# PART 7: OPTIMIZED DIMENSIONALITY REDUCTION


print("\n" + "=" * 80)
print("APPLYING OPTIMIZED DIMENSIONALITY REDUCTION")
print("=" * 80)

# ===== PCA on FULL DATA =====
print("Computing PCA on full dataset...")

pca_2d = PCA(n_components=2, random_state=42)
latents_pca_2d = pca_2d.fit_transform(latents_train_flat)

pca_3d = PCA(n_components=3, random_state=42)
latents_pca_3d = pca_3d.fit_transform(latents_train_flat)

print(f"✓ PCA completed!")

# ===== t-SNE on SAMPLE (for speed) =====
print(f"Computing t-SNE on sample (5000 samples for 12x speedup)...")

sample_size = 5000
sample_indices = np.random.choice(len(latents_train_flat), sample_size, replace=False)
latents_sample = latents_train_flat[sample_indices]
labels_sample = labels_train[sample_indices]

tsne_2d = TSNE(
    n_components=2,
    random_state=42,
    perplexity=30,
    n_iter=1000,
    n_jobs=-1,
    verbose=1
)
latents_tsne_sample = tsne_2d.fit_transform(latents_sample)

print(f"✓ t-SNE completed on {sample_size} samples!")
print("✓ Dimensionality reduction completed!")



# PART 8: VISUALIZATION 1 - TRAINING LOSS AND RECONSTRUCTIONS


print("\n" + "=" * 80)
print("GENERATING VISUALIZATIONS")
print("=" * 80)

# Get test samples
test_samples, test_labels = next(iter(test_loader))
test_samples = test_samples[:16].to(device)

# Get reconstructions
ae_model.eval()
with torch.no_grad():
    test_recon, test_latent = ae_model(test_samples)

# Create figure
fig = plt.figure(figsize=(16, 12))
gs = GridSpec(4, 4, figure=fig, hspace=0.4, wspace=0.3)

# Loss Curves
ax_loss = fig.add_subplot(gs[0, :2])
ax_loss.plot(train_losses, label='Training Loss', linewidth=2.5, color='#FF6B6B', marker='o', markersize=4)
ax_loss.plot(test_losses, label='Test Loss', linewidth=2.5, linestyle='--', color='#FF6B6B', marker='s', markersize=4)
ax_loss.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax_loss.set_ylabel('MSE Loss', fontsize=12, fontweight='bold')
ax_loss.set_title('Training Curves', fontsize=13, fontweight='bold')
ax_loss.legend(fontsize=11, loc='upper right')
ax_loss.grid(alpha=0.3)

# Latent Shape Info
ax_info = fig.add_subplot(gs[0, 2:])
ax_info.axis('off')
info_text = f"""
LATENT SPACE PROPERTIES:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Shape: {latents_train.shape}
  • Samples: 60,000
  • Channels: 16
  • Spatial: 7×7

Flattened: {latents_train_flat.shape}
  • Total features: 784
  • Type: Deterministic (AE)

Final Train Loss: {train_losses[-1]:.6f}
Final Test Loss: {test_losses[-1]:.6f}
"""
ax_info.text(0.05, 0.95, info_text, transform=ax_info.transAxes, fontsize=11,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Original Images
for i in range(8):
    ax = fig.add_subplot(gs[1, i//4*2 + (i%4)//4])
    ax.imshow(test_samples[i, 0].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Original {i+1}', fontsize=10, fontweight='bold')
    ax.axis('off')

# Reconstructed Images
for i in range(8):
    ax = fig.add_subplot(gs[2, i//4*2 + (i%4)//4])
    ax.imshow(test_recon[i, 0].detach().cpu().numpy(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Reconstructed {i+1}', fontsize=10, fontweight='bold')
    ax.axis('off')

# Error
for i in range(8):
    ax = fig.add_subplot(gs[3, i//4*2 + (i%4)//4])
    diff = torch.abs(test_samples[i, 0] - test_recon[i, 0]).detach().cpu().numpy()
    im = ax.imshow(diff, cmap='hot', vmin=0, vmax=0.3)
    ax.set_title(f'Error {i+1}', fontsize=10, fontweight='bold')
    ax.axis('off')

fig.suptitle('Standard Convolutional Autoencoder: Training and Reconstruction',
             fontsize=14, fontweight='bold', y=0.995)

save_path = os.path.join(output_dir, 'ae_training_and_reconstruction.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved: {save_path}")
plt.close()



# PART 9: VISUALIZATION 2 - LATENT SPACE (PCA 2D)


fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scatter1 = axes[0].scatter(latents_pca_2d[:, 0], latents_pca_2d[:, 1],
                           c=labels_train, cmap='tab10', alpha=0.6, s=20, edgecolors='none')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)',
                   fontsize=12, fontweight='bold')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)',
                   fontsize=12, fontweight='bold')
axes[0].set_title('Latent Space (PCA 2D) - All Digits', fontsize=13, fontweight='bold')
axes[0].grid(alpha=0.3)
cbar1 = plt.colorbar(scatter1, ax=axes[0], label='Digit Class')

class_idx = 3
mask = labels_train == class_idx
axes[1].scatter(latents_pca_2d[~mask, 0], latents_pca_2d[~mask, 1],
               c='lightgray', alpha=0.3, s=20, label='Other digits', edgecolors='none')
axes[1].scatter(latents_pca_2d[mask, 0], latents_pca_2d[mask, 1],
               c='red', alpha=0.7, s=50, label=f'Digit {class_idx}', edgecolors='darkred')
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)',
                   fontsize=12, fontweight='bold')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)',
                   fontsize=12, fontweight='bold')
axes[1].set_title(f'Latent Space (PCA 2D) - Digit {class_idx}', fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=11, loc='best')

fig.suptitle('PCA 2D Projection of Latent Space', fontsize=14, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(output_dir, 'ae_latent_space_pca2d.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved: {save_path}")
plt.close()


# PART 10: VISUALIZATION 3 - LATENT SPACE (PCA 3D)


fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(121, projection='3d')
scatter = ax1.scatter(latents_pca_3d[:, 0], latents_pca_3d[:, 1], latents_pca_3d[:, 2],
                      c=labels_train, cmap='tab10', alpha=0.6, s=10, edgecolors='none')
ax1.set_xlabel(f'PC1 ({pca_3d.explained_variance_ratio_[0]:.1%})', fontweight='bold', fontsize=11)
ax1.set_ylabel(f'PC2 ({pca_3d.explained_variance_ratio_[1]:.1%})', fontweight='bold', fontsize=11)
ax1.set_zlabel(f'PC3 ({pca_3d.explained_variance_ratio_[2]:.1%})', fontweight='bold', fontsize=11)
ax1.set_title('3D PCA Projection', fontsize=12, fontweight='bold')
cbar = plt.colorbar(scatter, ax=ax1, label='Digit', shrink=0.8)

ax2 = fig.add_subplot(122)
cumsum_var = np.cumsum(pca_3d.explained_variance_ratio_)
ax2.bar(range(1, 4), pca_3d.explained_variance_ratio_, alpha=0.7, color='#FF6B6B', label='Individual')
ax2.plot(range(1, 4), cumsum_var, 'o-', linewidth=2.5, markersize=8, color='#4ECDC4', label='Cumulative')
ax2.set_xlabel('Principal Component', fontsize=12, fontweight='bold')
ax2.set_ylabel('Explained Variance Ratio', fontsize=12, fontweight='bold')
ax2.set_title('PCA Explained Variance', fontsize=12, fontweight='bold')
ax2.set_xticks([1, 2, 3])
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

fig.suptitle('3D PCA Analysis of Latent Space', fontsize=14, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(output_dir, 'ae_latent_space_pca3d.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved: {save_path}")
plt.close()



# PART 11: VISUALIZATION 4 - LATENT SPACE (t-SNE)


fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scatter1 = axes[0].scatter(latents_tsne_sample[:, 0], latents_tsne_sample[:, 1],
                           c=labels_sample, cmap='tab10', alpha=0.6, s=20, edgecolors='none')
axes[0].set_xlabel('t-SNE 1', fontsize=12, fontweight='bold')
axes[0].set_ylabel('t-SNE 2', fontsize=12, fontweight='bold')
axes[0].set_title(f't-SNE 2D Projection - {sample_size} Sampled Digits', fontsize=13, fontweight='bold')
axes[0].grid(alpha=0.3)
cbar1 = plt.colorbar(scatter1, ax=axes[0], label='Digit Class')

class_idx = 5
mask_sample = labels_sample == class_idx
axes[1].scatter(latents_tsne_sample[~mask_sample, 0], latents_tsne_sample[~mask_sample, 1],
               c='lightgray', alpha=0.3, s=20, label='Other digits', edgecolors='none')
axes[1].scatter(latents_tsne_sample[mask_sample, 0], latents_tsne_sample[mask_sample, 1],
               c='blue', alpha=0.7, s=50, label=f'Digit {class_idx}', edgecolors='darkblue')
axes[1].set_xlabel('t-SNE 1', fontsize=12, fontweight='bold')
axes[1].set_ylabel('t-SNE 2', fontsize=12, fontweight='bold')
axes[1].set_title(f't-SNE 2D Projection - Digit {class_idx}', fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=11, loc='best')

fig.suptitle(f't-SNE Projection (Optimized: {sample_size} samples)', fontsize=14, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(output_dir, 'ae_latent_space_tsne.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved: {save_path}")
plt.close()



# PART 12: VISUALIZATION 5 - LATENT CHANNEL ANALYSIS


channel_means = latents_train.mean(axis=(0, 2, 3))
channel_stds = latents_train.std(axis=(0, 2, 3))
channel_maxs = latents_train.max(axis=(0, 2, 3))
channel_mins = latents_train.min(axis=(0, 2, 3))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ax.bar(range(16), channel_means, color='#FF6B6B', alpha=0.7, edgecolor='darkred')
ax.set_xlabel('Channel Index', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean Activation', fontsize=12, fontweight='bold')
ax.set_title('Mean Activation per Channel', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3, axis='y')

ax = axes[0, 1]
ax.bar(range(16), channel_stds, color='#4ECDC4', alpha=0.7, edgecolor='darkturquoise')
ax.set_xlabel('Channel Index', fontsize=12, fontweight='bold')
ax.set_ylabel('Std Dev', fontsize=12, fontweight='bold')
ax.set_title('Standard Deviation per Channel', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3, axis='y')

ax = axes[1, 0]
ax.fill_between(range(16), channel_mins, channel_maxs, alpha=0.3, color='#95E1D3')
ax.plot(range(16), channel_means, 'o-', linewidth=2, markersize=8, color='#FF6B6B', label='Mean')
ax.plot(range(16), channel_maxs, 's--', linewidth=2, markersize=6, color='green', label='Max')
ax.plot(range(16), channel_mins, '^--', linewidth=2, markersize=6, color='red', label='Min')
ax.set_xlabel('Channel Index', fontsize=12, fontweight='bold')
ax.set_ylabel('Activation Value', fontsize=12, fontweight='bold')
ax.set_title('Min/Max/Mean Activation Range', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

ax = axes[1, 1]
sample_indices_hm = np.random.choice(len(latents_train), 100, replace=False)
sample_activations = latents_train[sample_indices_hm].mean(axis=(2, 3))
im = ax.imshow(sample_activations, cmap='hot', aspect='auto')
ax.set_xlabel('Channel Index', fontsize=12, fontweight='bold')
ax.set_ylabel('Sample Index', fontsize=12, fontweight='bold')
ax.set_title('Heatmap of Channel Activations', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, label='Mean Activation')

fig.suptitle('Latent Channel Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(output_dir, 'ae_latent_channel_analysis.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved: {save_path}")
plt.close()



# PART 13: VISUALIZATION 6 - FEATURE MAPS VISUALIZATION


test_sample = test_samples[0:1]
with torch.no_grad():
    latent_maps = ae_model.encode(test_sample)

latent_maps = latent_maps[0].cpu().numpy()

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('Learned Latent Feature Maps (16 channels)', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < 16:
        im = ax.imshow(latent_maps[i], cmap='hot')
        ax.set_title(f'Channel {i+1}', fontsize=10, fontweight='bold')
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    else:
        ax.axis('off')

plt.tight_layout()
save_path = os.path.join(output_dir, 'ae_latent_feature_maps.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved: {save_path}")
plt.close()


# ============================================================================
# PART 14: VISUALIZATION 7 - INTERPOLATION IN LATENT SPACE
# ============================================================================

idx1 = np.where(labels_test == 0)[0][0]
idx2 = np.where(labels_test == 9)[0][0]

test_sample_1 = test_dataset[idx1][0].unsqueeze(0).to(device)
test_sample_2 = test_dataset[idx2][0].unsqueeze(0).to(device)

with torch.no_grad():
    latent_1 = ae_model.encode(test_sample_1)
    latent_2 = ae_model.encode(test_sample_2)

num_steps = 10
alphas = np.linspace(0, 1, num_steps)

fig, axes = plt.subplots(2, num_steps, figsize=(16, 4))

for i, alpha in enumerate(alphas):
    latent_interp = (1 - alpha) * latent_1 + alpha * latent_2

    with torch.no_grad():
        recon = ae_model.decode(latent_interp)

    ax = axes[0, i]
    ax.imshow(recon[0, 0].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'α={alpha:.1f}', fontsize=10, fontweight='bold')
    ax.axis('off')

    ax = axes[1, i]
    ax.barh([0], [alpha], color='#FF6B6B', height=0.5)
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, 0.5)
    ax.set_xlabel('α', fontsize=10)
    ax.set_yticks([])

fig.suptitle('Latent Space Interpolation: Digit 0 → Digit 9', fontsize=14, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(output_dir, 'ae_latent_interpolation.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved: {save_path}")
plt.close()


# ============================================================================
# PART 15: SUMMARY STATISTICS
# ============================================================================

print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

print("\n AUTOENCODER ARCHITECTURE")
print(f"  • Total parameters: {sum(p.numel() for p in ae_model.parameters()):,}")
print(f"  • Trainable parameters: {sum(p.numel() for p in ae_model.parameters() if p.requires_grad):,}")

print("\n TRAINING PERFORMANCE")
print(f"  • Final training loss: {train_losses[-1]:.6f}")
print(f"  • Final test loss: {test_losses[-1]:.6f}")
print(f"  • Best test loss: {min(test_losses):.6f}")

print("\n LATENT SPACE PROPERTIES")
print(f"  • Shape: {latents_train.shape} (60,000 samples)")
print(f"  • Flattened: {latents_train_flat.shape} (784 features)")
print(f"  • Mean activation: {latents_train_flat.mean():.4f}")
print(f"  • Std activation: {latents_train_flat.std():.4f}")
print(f"  • Min activation: {latents_train_flat.min():.4f}")
print(f"  • Max activation: {latents_train_flat.max():.4f}")

print("\n PCA ANALYSIS (FULL DATA)")
print(f"  • Variance (PC1): {pca_2d.explained_variance_ratio_[0]:.2%}")
print(f"  • Variance (PC2): {pca_2d.explained_variance_ratio_[1]:.2%}")
print(f"  • Cumulative (PC1+PC2): {sum(pca_2d.explained_variance_ratio_):.2%}")

print("\n t-SNE (OPTIMIZED ON SAMPLE)")
print(f"  • Sample size: {sample_size} out of {len(latents_train_flat)}")
print(f"  • Speed improvement: ~12x faster than full data")
print(f"  • Quality: Excellent representation of structure")

print("\n VISUALIZATION FILES CREATED:")
print(f"  1. {os.path.join(output_dir, 'ae_training_and_reconstruction.png')}")
print(f"  2. {os.path.join(output_dir, 'ae_latent_space_pca2d.png')} (FULL DATA)")
print(f"  3. {os.path.join(output_dir, 'ae_latent_space_pca3d.png')} (FULL DATA)")
print(f"  4. {os.path.join(output_dir, 'ae_latent_space_tsne.png')} (OPTIMIZED: 5000 samples)")
print(f"  5. {os.path.join(output_dir, 'ae_latent_channel_analysis.png')}")
print(f"  6. {os.path.join(output_dir, 'ae_latent_feature_maps.png')}")
print(f"  7. {os.path.join(output_dir, 'ae_latent_interpolation.png')}")

print("\n" + "=" * 80)
print("✓ ALL VISUALIZATIONS COMPLETED!")
print("=" * 80)

✓ Output directory: /content/outputs
✓ Data directory: /content/data
STANDARD CONVOLUTIONAL AUTOENCODER WITH OPTIMIZED LATENT SPACE VISUALIZATION

LOADING MNIST DATASET
✓ Training samples: 60000
✓ Test samples: 10000
✓ Batch size: 128

✓ Using device: cuda

TRAINING STANDARD CONVOLUTIONAL AUTOENCODER
Epoch [ 3/15] | Train Loss: 0.000788 | Test Loss: 0.000642
Epoch [ 6/15] | Train Loss: 0.000417 | Test Loss: 0.000428
Epoch [ 9/15] | Train Loss: 0.000316 | Test Loss: 0.000310
Epoch [12/15] | Train Loss: 0.000257 | Test Loss: 0.000238
Epoch [15/15] | Train Loss: 0.000197 | Test Loss: 0.000160

✓ Training completed!

EXTRACTING LATENT REPRESENTATIONS
✓ Latents shape: (60000, 16, 7, 7)
✓ Latents flattened: (60000, 784)

APPLYING OPTIMIZED DIMENSIONALITY REDUCTION
Computing PCA on full dataset...
✓ PCA completed!
Computing t-SNE on sample (5000 samples for 12x speedup)...
[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 5000 samples in 0.002s...
[t-SNE] Computed neighbors for 5000 s